In [29]:
import pandas as pd
import spacy

In [30]:
# lendo o dataset
df = pd.read_csv('files/stocks.tsv', sep='\t')
df.head()

,Symbol,CompanyName,Industry,MarketCap
0,A,Agilent Technologies,Life Sciences Tools & Services,53.65B
1,AA,Alcoa,Metals & Mining,9.25B
2,AAC,Ares Acquisition,Shell Companies,1.22B
3,AACG,ATA Creativity Global,Diversified Consumer Services,90.35M
4,AADI,Aadi Bioscience,Pharmaceuticals,104.85M


In [31]:
# transformando as colunas em listas
symbols = df.Symbol.tolist()
companies = df.CompanyName.tolist()
print(symbols[:10])

['A', 'AA', 'AAC', 'AACG', 'AADI', 'AAIC', 'AAL', 'AAMC', 'AAME', 'AAN']


In [ ]:
# lendo dataset de index das empresas no mercado de ações
df2 = pd.read_csv('files/indexes.tsv', sep='\t')
df2.head()

,IndexName,IndexSymbol
0,Dow Jones Industrial Average,DJIA
1,Dow Jones Transportation Average,DJT
2,Dow Jones Utility Average Index,DJU
3,NASDAQ 100 Index (NASDAQ Calculation),NDX
4,NASDAQ Composite Index,COMP


In [ ]:
# pegando os dados como listas
indexes = df2.IndexName.tolist()
index_symbols = df2.IndexSymbol.tolist()

In [44]:
# carregando dataset da bolsa de valores
df3 = pd.read_csv('files/stock_exchanges.tsv', sep='\t')
df3.head()

,BloombergExchangeCode,BloombergCompositeCode,Country,Description,ISOMIC,Google Prefix,EODcode,NumStocks
0,AF,AR,Argentina,Bolsa de Comercio de Buenos Aires,XBUE,NaN,BA,12
1,AO,AU,Australia,National Stock Exchange of Australia,XNEC,NaN,NaN,1
2,AT,AU,Australia,Asx - All Markets,XASX,ASX,AU,875
3,AV,NaN,Austria,Wiener Boerse Ag,XWBO,VIE,VI,38
4,BI,NaN,Bahrain,Bahrain Bourse,XBAH,NaN,NaN,4


In [ ]:
# lista contendo 3 colunas juntas (ISOMIC, google prefix, e a descrição)
exchanges = df3.ISOMIC.tolist()+df3['Google Prefix'].tolist()+df3.Description.tolist()
print(exchanges[-10:])

['NYSE American', 'CBOE BATS BZX', 'New York Stock Exchange', 'NYSE Arca', 'NASDAQ Global Market', 'NASDAQ Capital Market', 'OTC markets', 'NASDAQ Global Select', 'Hanoi Stock Exchange', 'Hochiminh Stock Exchange']


In [ ]:
# criterios de paradas
stops = ['two']
# pipeline vazio em ingles
nlp = spacy.blank('en')
# adicionando ruler no pipeline
ruler = nlp.add_pipe('entity_ruler')
# letras do alfabeto
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
# lista de padrões a serem reconhecidos
patterns = []

# criando os dicionarios de padrões para reconhecer nome da empresa e simbolos
for symbol in symbols:
    patterns.append({'label': 'STOCK', 'pattern': symbol})

    # fazendo com que reconheça as empresas que tem um ponto + letra depois do simbolo
    for l in letters:
        patterns.append({'label': 'STOCK', 'pattern': symbol+f'.{l}'})

# reconhecendo as empresas
for company in companies:
    if company not in stops:
        patterns.append({'label': 'COMPANY', 'pattern': company})

# reconhecendo os indexes das empresas no mercado de ações
for index in indexes:
    patterns.append({'label': 'INDEX', 'pattern': index})
    words = index.split()
    patterns.append({'label': 'INDEX', 'pattern': ' '.join(words[:2])})

# reconhecendo os simbolos dos indexes das empresas
for index in index_symbols:
    patterns.append({'label': 'INDEX', 'pattern': index})

# reconhecendo as entidades da bolsa de valores
for e in exchanges:
    patterns.append({'label': 'STOCK_EXCHANGE', 'pattern': e})

# adicionando os padrões no ruler
ruler.add_patterns(patterns)

In [35]:
# texto de uma noticia
with open('files/text_example.txt') as f:
    text = f.read()

In [49]:
# criando um doc
doc = nlp(text)

In [50]:
# listando as entidades reconhecidas
for ent in doc.ents:
    print(ent.text, ent.label_)

Apple COMPANY
Apple COMPANY
AAPL.O STOCK
Apple COMPANY
Nasdaq COMPANY
S&P 500 INDEX
S&P 500 INDEX
ET STOCK
Dow Jones Industrial Average INDEX
S&P 500 INDEX
Nasdaq COMPANY
S&P 500 INDEX
JD.com COMPANY
TME.N STOCK
NIO.N STOCK
Kroger COMPANY
KR.N STOCK
NYSE STOCK_EXCHANGE
Nasdaq COMPANY
Nasdaq COMPANY


In [38]:
from spacy import displacy

In [ ]:
# mostrando as entidades de forma renderizada
displacy.render(doc, style='ent')